# Gruppe 4 Abgabe

## Import der wichtigsten Pakete

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

from scipy import stats
from statsmodels.stats.proportion import (
    proportion_confint,
    proportions_ztest,
    confint_proportions_2indep,
)

## Einlesen der Datensätze

In [16]:
#Plz als Text einlesen damit führende Nullen erhalten bleiben

plz = {"customer_zip_code_prefix":str,
        "seller_zip_code_prefix":str,
        "geolocation_zip_code_prefix":str}


df_categories = pd.read_csv("data/categories.csv")
df_customers = pd.read_csv("data/customers.csv", dtype =plz)
df_geolocation = pd.read_csv("data/geolocation.csv", dtype=plz)
df_items = pd.read_csv("data/items.csv")
df_orders= pd.read_csv("data/orders.csv")
df_payments = pd.read_csv("data/payments.csv")
df_pilot_support = pd.read_csv("data/pilot_support.csv")
df_products = pd.read_csv("data/products.csv")
df_reviews = pd.read_csv("data/reviews.csv")
df_sellers = pd.read_csv("data/sellers.csv", dtype =plz)
df_service_experiment = pd.read_csv("data/service_experiment.csv")
df_service_observation = pd.read_csv("data/service_observation.csv")

# 2.1 a) Prüfung und Bereinigung der Datensätze


## Funktion um Tabellen einheitlich zu Prüfen

In [17]:
def pruefe_tabelle(df, name, schluessel, kategorien=None, fremdschluessel=None):

    # 1. Struktur und Datentypen
    print("Shape:", df.shape)
    df.info()

    # 2. Fehlende Werte (absolut und in %)
    na = pd.DataFrame({'fehlend': df.isna().sum(),
                       'prozent': (df.isna().mean() * 100).round(2)})
    print("\nFehlende Werte:\n", na[na['fehlend'] > 0] if na['fehlend'].any() else "keine")

    # 3. Duplikate
    print("\nDoppelte Zeilen:", df.duplicated().sum())
    print(f"Doppelte Schlüssel {schluessel}:", df.duplicated(subset=schluessel).sum())

    # 4. Wertebereiche der Zahlenspalten
    zahlen = df.select_dtypes(include='number')
    if not zahlen.empty:
        print("\nWertebereiche:\n", zahlen.describe().T)

    # 5. Kategorien
    for spalte in (kategorien or []):
        print(f"\nKategorien in {spalte}:\n", df[spalte].value_counts(dropna=False))

    # 6. Bezüge zu anderen Tabellen
    for spalte, (ziel_df, ziel_spalte) in (fremdschluessel or {}).items():
        werte = df[spalte].dropna()
        fehlend = (~werte.isin(ziel_df[ziel_spalte])).sum()
        print(f"\n{spalte} nicht in Zieltabelle ({ziel_spalte}): {fehlend} von {len(werte)}")

#Umwandlung in der Zeitspalten in DateTime-Format
def umwandlung_zeitspalte(df, spalten, name):

    df = df.copy()
    na_vorher = df[spalten].isna().sum()

    for spalte in spalten:
        df[spalte] = pd.to_datetime(df[spalte], errors='coerce')

    neu_fehlend = df[spalten].isna().sum() - na_vorher
    print(f"{name}: neue fehlende Werte durch Umwandlung:\n{neu_fehlend}")

    return df

## Orders.csv Datei:

In [18]:
#Übersicht der Daten
df_orders.head(10)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01 00:00:00
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaN,NaN,2017-05-09 00:00:00
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07 00:00:00
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06 00:00:00
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23 00:00:00


In [19]:
df_orders = umwandlung_zeitspalte(df_orders, ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"], "orders")


orders: neue fehlende Werte durch Umwandlung:
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
dtype: int64


In [21]:
pruefe_tabelle(df_orders, "orders", ["order_id"],
                kategorien=["order_status"],
                fremdschluessel={"customer_id": (df_customers, "customer_id")})

#Zeitraum der Zeitspalten
zeitspalten = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]
zeitspalten_df = df_orders[zeitspalten]
df_orders[zeitspalten].agg(['min', 'max']).T
zeitspalten_min_max = df_orders[zeitspalten].agg(['min', 'max']).T
zeitspalten_min_max.columns = ['min', 'max']
print(zeitspalten_min_max)
#Dokumentation: Daten stimmen in dem Zeitraum überein

#Zeitliche Reihenfolge der Orders prüfen: Kauf-Freigabe -Übergabe an Transporteur - Lieferung

print("Anzahl der Orders, bei denen die Freigabe vor dem Kauf erfolgt ist:")
print((df_orders["order_approved_at"] < df_orders["order_purchase_timestamp"]).sum())
print("Anzahl der Orders, bei denen die Übergabe an den Transporteur vor der Freigabe erfolgt ist:")
print((df_orders["order_delivered_carrier_date"] < df_orders["order_approved_at"]).sum())
print("Anzahl der Orders, bei denen die Lieferung vor der Übergabe an den Transporteur erfolgt ist:")
print((df_orders["order_delivered_customer_date"] < df_orders["order_delivered_carrier_date"]).sum())
print("Transporteur vor Kauf:",(df_orders["order_delivered_carrier_date"] < df_orders["order_purchase_timestamp"]).sum())
print("Lieferung vor Kauf:",(df_orders["order_delivered_customer_date"] < df_orders["order_purchase_timestamp"]).sum())


Shape: (99441, 8)
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 13.0 MB

Fehlende Werte:
                                fehlend  prozent
order_approved_at                  160     0.16
order_delivered_carrier_date      1783     1.

## customer.csv Datei:

In [22]:
df_customers.head(10)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
5,879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
6,fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,04534,sao paulo,SP
7,5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG
8,5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR
9,4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG


In [23]:
pruefe_tabelle(
    df_customers, "customers", ["customer_id"],
    kategorien=["customer_state"]
)

# PLZ-Präfixe: Wie oft kommt welche Länge vor?
print(df_customers["customer_zip_code_prefix"].str.len().value_counts())

# Wiederkehrende Kunden über customer_unique_id
bestellungen_je_kunde = df_customers["customer_unique_id"].value_counts()
print("Kunden insgesamt:", len(bestellungen_je_kunde))
print("Kunden mit mehr als einer Bestellung:", (bestellungen_je_kunde > 1).sum())
print("Anteil:", round((bestellungen_je_kunde > 1).mean() * 100, 2), "%")
print("Maximale Bestellungen eines Kunden:", bestellungen_je_kunde.max())

Shape: (99441, 5)
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  str  
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: str(5)
memory usage: 11.5 MB

Fehlende Werte:
 keine

Doppelte Zeilen: 0
Doppelte Schlüssel ['customer_id']: 0

Kategorien in customer_state:
 customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int6

## items.csv

In [24]:
df_items.head(10)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,ce02b2c6e70d5b2fe24d103952d389bf,1,70d17ed46123e5b68dff23e113415d5a,3d871de0142ce09b7081e2b9d1733cb1,2017-08-03 13:45:11,54.90,12.72
1,4886c605fd1f7edd568a936814a015b7,1,bb09b0b5f3668effd704b1b2a83cae9b,6560211a19b47992c3666cc44a7e94c0,2017-12-22 02:31:10,189.00,8.58
2,2abac84637c5e0ddd2923f22ba61a96a,1,4ae634441e444ca4bc85903cafe98d73,92eb0f42c21942b6552362b9b114707d,2017-04-05 01:22:35,25.00,16.05
3,906dfb858a6afcaea1fb5cc0ed730147,1,33e77eecc23c25a65d10f627e5053617,7c67e1448b00f6e969d365cea6b010ab,2017-11-01 02:49:41,114.94,28.16
4,dcc495566353215aa81cab568f8fb2bf,1,6fd08d44046ab994b96ff38ad6fcfba1,d9a84e1403de8da0c3aa531d6d108ba6,2018-03-30 16:08:15,39.90,18.23
5,f8b6b27efdf0f73f9e9d978fc503c45f,1,2647221a6a477afe2d0268b378f1fdcd,d2374cbcbb3ca4ab1086534108cc3ab7,2018-07-10 17:31:38,39.90,19.47
6,d1c3d911ad830cab7d16ad8acfc36a46,2,05f0fe07929d35be0748fac56eecbeb2,c70c1b0d8ca86052f45a432a38b73958,2018-05-15 15:12:12,110.32,15.65
7,0ee0c320d2ceda401e28f8bc01e2fde2,1,5f24849289916bf82378a061a131fde9,be9a160a9011f627c3e121d076c1ec7b,2017-03-31 11:45:13,289.00,40.65
8,a7765f04ecc5f2e0683f08a97ee8174e,1,77d1f94d7eeaaeb36736fd1049e79eb3,8e6cc767478edae941d9bd9eb778d77a,2017-02-02 19:08:00,64.90,18.00
9,1c300b10263fefc0edc45c99921586a7,1,20a2613eb3b1c3819e733de3e91644de,1025f0e2d44d7041d6cf58b6550e0bfa,2018-04-02 21:15:25,35.00,8.29


In [25]:
df_items= umwandlung_zeitspalte(df_items, ["shipping_limit_date"], "items")

items: neue fehlende Werte durch Umwandlung:
shipping_limit_date    0
dtype: int64


In [26]:
pruefe_tabelle(
    df_items, "items", ["order_id", "order_item_id"],
    fremdschluessel={"order_id": (df_orders, "order_id"),
                     "product_id": (df_products, "product_id"),
                     "seller_id": (df_sellers, "seller_id")}
)


Shape: (112875, 7)
<class 'pandas.DataFrame'>
RangeIndex: 112875 entries, 0 to 112874
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112875 non-null  str           
 1   order_item_id        112875 non-null  int64         
 2   product_id           112875 non-null  str           
 3   seller_id            112875 non-null  str           
 4   shipping_limit_date  112875 non-null  datetime64[us]
 5   price                112875 non-null  float64       
 6   freight_value        112875 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 16.4 MB

Fehlende Werte:
 keine

Doppelte Zeilen: 225
Doppelte Schlüssel ['order_id', 'order_item_id']: 225

Wertebereiche:
                   count        mean         std   min    25%    50%     75%  \
order_item_id  112875.0    1.197927    0.705163  1.00   1.00   1.00    1.00   
price  

In [27]:
#Doppelte Zeilen entfernen, die den gleichen Schlüssel haben und alle Spalten gleich sind
df_items_clean = df_items.drop_duplicates().reset_index(drop=True)
assert not df_items_clean.duplicated(subset=["order_id", "order_item_id"]).any()
print("Zeilen vorher:", len(df_items), "| nachher:", len(df_items_clean))

#Lückenlose Positionsnummern
pos = df_items_clean.groupby("order_id")["order_item_id"].agg(["min", "max", "count"])
luecken = (pos["min"] != 1) | (pos["max"] != pos["count"])
print("Bestellungen mit Lücken in der Positionsnummer:", luecken.sum())

#Zeitraum von shipping_limit_date
print("Zeitraum von shipping_limit_date:")
print("Von:", df_items_clean["shipping_limit_date"].min())
print("Bis:", df_items_clean["shipping_limit_date"].max())

Zeilen vorher: 112875 | nachher: 112650
Bestellungen mit Lücken in der Positionsnummer: 0
Zeitraum von shipping_limit_date:
Von: 2016-09-19 00:15:34
Bis: 2020-04-09 22:35:08


## Sellers

In [28]:
df_sellers.head(10)

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
5,c240c4061717ac1806ae6ee72be3533b,20920,rio de janeiro,RJ
6,e49c26c3edfa46d227d5121a6b6e4d37,55325,brejao,PE
7,1b938a7ec6ac5061a66a3766e0e75f90,16304,penapolis,SP
8,768a86e36ad6aae3d03ee3c6433d61df,01529,sao paulo,SP
9,ccc4bbb5f32a6ab2b7066a4130f114e3,80310,curitiba,PR


In [29]:
pruefe_tabelle(
    df_sellers, "sellers", ["seller_id"],
    kategorien=["seller_state"]
)

# PLZ-Präfixe: Wie oft kommt welche Länge vor?
print(df_sellers["seller_zip_code_prefix"].str.len().value_counts())

#Händler ohne Verkäufe
df_sellers_no_sales = df_sellers[~df_sellers["seller_id"].isin(df_items_clean["seller_id"])]  # ~ wählt die aus wo nicht Verkaufen
print("Händler ohne Verkäufe:", len(df_sellers_no_sales))

Shape: (3095, 4)
<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   str  
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: str(4)
memory usage: 245.5 KB

Fehlende Werte:
 keine

Doppelte Zeilen: 0
Doppelte Schlüssel ['seller_id']: 0

Kategorien in seller_state:
 seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
CE      13
PE       9
PB       6
RN       5
MS       5
MT       4
RO       2
SE       2
AC       1
PI       1
MA       1
AM       1
PA       1
Name: count, dtype: int64
seller_zip_code_prefix
5    3095
Name: count, dtype: int64
Händler ohne Verkäufe: 0


## Reviews.csv

In [30]:
df_reviews.head(10)

,review_id,order_id,review_score,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,2018-03-01 00:00:00,2018-03-02 10:26:53
5,15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,2018-04-13 00:00:00,2018-04-16 00:39:37
6,07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,2017-07-16 00:00:00,2017-07-18 19:30:34
7,7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,2018-08-14 00:00:00,2018-08-14 21:36:06
8,a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,2017-05-17 00:00:00,2017-05-18 12:05:37
9,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,2018-05-22 00:00:00,2018-05-23 16:45:47


In [31]:
df_reviews = umwandlung_zeitspalte(df_reviews, ["review_creation_date", "review_answer_timestamp"], "reviews")

reviews: neue fehlende Werte durch Umwandlung:
review_creation_date       0
review_answer_timestamp    0
dtype: int64


In [32]:
pruefe_tabelle(
    df_reviews, "reviews", ["review_id"],
    kategorien=["review_score"],
    fremdschluessel={"order_id": (df_orders, "order_id")})

#Mehrere Reviews pro Bestellung
mehrere_reviews = df_reviews.groupby("order_id").size()
print("Bestellungen mit mehreren Reviews:", (mehrere_reviews > 1).sum())

#Zeitlogik liegt creation date vor answer timestamp

print("Anzahl der Reviews, bei denen die Antwort vor der Erstellung erfolgt ist:")
print((df_reviews["review_answer_timestamp"] < df_reviews["review_creation_date"]).sum())

#Liegt die Anfrage nach dem Kaufdatum?
print("Anzahl der Reviews, bei denen die Erstellung vor dem Kaufdatum erfolgt ist:")
kaufzeit = df_reviews["order_id"].map(df_orders.set_index("order_id")["order_purchase_timestamp"])
print((df_reviews["review_creation_date"] < kaufzeit.dt.normalize()).sum())  #normalize setzt zeit auf 0:00 damit tag verglichen wird

Shape: (99224, 5)
<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                99224 non-null  str           
 1   order_id                 99224 non-null  str           
 2   review_score             99224 non-null  int64         
 3   review_creation_date     99224 non-null  datetime64[us]
 4   review_answer_timestamp  99224 non-null  datetime64[us]
dtypes: datetime64[us](2), int64(1), str(2)
memory usage: 9.8 MB

Fehlende Werte:
 keine

Doppelte Zeilen: 0
Doppelte Schlüssel ['review_id']: 814

Wertebereiche:
                 count      mean       std  min  25%  50%  75%  max
review_score  99224.0  4.086421  1.347579  1.0  4.0  5.0  5.0  5.0

Kategorien in review_score:
 review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64

order_id nicht in Zieltabelle (orde

## Products.csv

In [33]:
df_products.head(10)

,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_photos_qty
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,225.0,16.0,10.0,14.0,1.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,1000.0,30.0,18.0,20.0,1.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,154.0,18.0,9.0,15.0,1.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,371.0,26.0,4.0,26.0,1.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,625.0,20.0,17.0,13.0,4.0
5,41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,200.0,38.0,5.0,11.0,1.0
6,732bd381ad09e530fe0a5f457d81becb,cool_stuff,18350.0,70.0,24.0,44.0,4.0
7,2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,900.0,40.0,8.0,40.0,2.0
8,37cc742be07708b53a98702e77a21a02,eletrodomesticos,400.0,27.0,13.0,17.0,1.0
9,8c92109888e8cdf9d66dc7e463025574,brinquedos,600.0,17.0,10.0,12.0,1.0


In [34]:
pruefe_tabelle(
    df_products, "products", ["product_id"],
    fremdschluessel={"product_category_name": (df_categories, "product_category_name")})


Shape: (32951, 7)
<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   product_id             32951 non-null  str    
 1   product_category_name  32341 non-null  str    
 2   product_weight_g       32619 non-null  float64
 3   product_length_cm      32949 non-null  float64
 4   product_height_cm      32949 non-null  float64
 5   product_width_cm       32949 non-null  float64
 6   product_photos_qty     32341 non-null  float64
dtypes: float64(5), str(2)
memory usage: 3.2 MB

Fehlende Werte:
                        fehlend  prozent
product_category_name      610     1.85
product_weight_g           332     1.01
product_length_cm            2     0.01
product_height_cm            2     0.01
product_width_cm             2     0.01
product_photos_qty         610     1.85

Doppelte Zeilen: 0
Doppelte Schlüssel ['product_id']: 0

Wertebereiche

In [35]:
#Prüfung der ungültigen Kategorienamen
gueltige_kategorien = set(df_categories["product_category_name"])

kategorie = df_products["product_category_name"]
ungueltig = kategorie.notna() & ~kategorie.isin(gueltige_kategorien)

print("Ungültige Kategorienamen:", ungueltig.sum())
# repr macht Leerzeichen am Ende sichtbar
print(kategorie[ungueltig].map(repr).value_counts().head(15).to_string())

#Extreme Gewichte überprüfen:
gewicht = df_products["product_weight_g"]

print("Produkte mit Gewicht 0 g:", (gewicht == 0).sum())

# Die 10 schwersten Produkte mit ihren Maßen
massspalten = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
display(df_products.nlargest(10, "product_weight_g")[massspalten])

# Gibt es eine Lücke in der Verteilung? Alle Gewichte zwischen 30 kg und 200 kg:
print("Gewichte zwischen 30 kg und 200 kg:")
print(sorted(gewicht[(gewicht > 30_000) & (gewicht < 200_000)].unique()))

#dichte als Plausibilitätsprüfung: Gewicht / Volumen
volumen = (df_products["product_length_cm"]
           * df_products["product_height_cm"]
           * df_products["product_width_cm"])
dichte = df_products["product_weight_g"] / volumen

print("Median-Dichte aller Produkte:", round(dichte.median(), 3))
print("Produkte mit Dichte über 8 (schwerer als Stahl):", (dichte > 8).sum())

#Muster der Fehlenden Werte
fehlt_kategorie = df_products["product_category_name"].isna()
fehlt_fotos = df_products["product_photos_qty"].isna()
fehlt_gewicht = df_products["product_weight_g"].isna()
fehlt_masse = df_products["product_length_cm"].isna()

print("Kategorie und Fotoanzahl fehlen bei denselben Produkten:", (fehlt_kategorie == fehlt_fotos).all())
print("Gewicht fehlt:", fehlt_gewicht.sum(), "| davon fehlen auch die Maße:", (fehlt_gewicht & fehlt_masse).sum())

Ungültige Kategorienamen: 330
product_category_name
'CAMA_MESA_BANHO'            10
'cama_mesa_banho '           10
'esporte_lazer '             10
'MOVEIS_DECORACAO'            9
'moveis_decoracao '           9
'ESPORTE_LAZER'               9
'beleza_saude '               8
'BELEZA_SAUDE'                8
'UTILIDADES_DOMESTICAS'       8
'utilidades_domesticas '      8
'automotivo '                 7
'informatica_acessorios '     6
'AUTOMOTIVO'                  6
'BRINQUEDOS'                  5
'INFORMATICA_ACESSORIOS'      5
Produkte mit Gewicht 0 g: 4


,product_weight_g,product_length_cm,product_height_cm,product_width_cm
24236,6900000.0,70.0,15.0,33.0
4716,6450000.0,20.0,20.0,20.0
8023,5900000.0,35.0,12.0,20.0
16667,2100000.0,55.0,25.0,35.0
21592,1950000.0,50.0,8.0,34.0
4969,1700000.0,26.0,9.0,20.0
6599,1600000.0,43.0,8.0,33.0
22358,1600000.0,42.0,11.0,36.0
18777,1450000.0,32.0,12.0,30.0
8063,1100000.0,42.0,20.0,31.0


Gewichte zwischen 30 kg und 200 kg:
[np.float64(40425.0), np.float64(150000.0)]
Median-Dichte aller Produkte: 0.117
Produkte mit Dichte über 8 (schwerer als Stahl): 58
Kategorie und Fotoanzahl fehlen bei denselben Produkten: True
Gewicht fehlt: 332 | davon fehlen auch die Maße: 2


In [36]:
#Vereineheitlichen der Kategorien
df_products_clean = df_products.copy()

# Schritt 1: Leerzeichen entfernen und kleinschreiben (NaN bleibt NaN)
df_products_clean["product_category_name"] = (
    df_products_clean["product_category_name"].str.strip().str.lower())

# Schritt 2: Buchstabendreher korrigieren.
# Idee: Sortiert man die Buchstaben eines Namens, sind Dreher und richtiger Name gleich.
def buchstaben(text):
    return "".join(sorted(text))

nachschlagen = {buchstaben(k): k for k in gueltige_kategorien}
assert len(nachschlagen) == len(gueltige_kategorien)   # Zuordnung ist eindeutig

def korrigiere_dreher(name):
    if pd.isna(name) or name in gueltige_kategorien:
        return name
    return nachschlagen.get(buchstaben(name), name)

df_products_clean["product_category_name"] = df_products_clean["product_category_name"].map(korrigiere_dreher)

# Kontrolle: jede vorhandene Kategorie ist jetzt gültig
neu = df_products_clean["product_category_name"]
assert not (neu.notna() & ~neu.isin(gueltige_kategorien)).any()

# Beleg: Welche Schreibweise wurde worauf abgebildet?
alt = df_products["product_category_name"]
geaendert = alt.notna() & (alt != neu)
zuordnung = pd.DataFrame({"vorher": alt[geaendert], "nachher": neu[geaendert]})
print("Korrigierte Produkte:", geaendert.sum())
display(zuordnung.value_counts().reset_index(name="Anzahl").head(15))


Korrigierte Produkte: 330


,vorher,nachher,Anzahl
0,CAMA_MESA_BANHO,cama_mesa_banho,10
1,cama_mesa_banho,cama_mesa_banho,10
2,esporte_lazer,esporte_lazer,10
3,MOVEIS_DECORACAO,moveis_decoracao,9
4,moveis_decoracao,moveis_decoracao,9
5,ESPORTE_LAZER,esporte_lazer,9
6,beleza_saude,beleza_saude,8
7,BELEZA_SAUDE,beleza_saude,8
8,UTILIDADES_DOMESTICAS,utilidades_domesticas,8
9,utilidades_domesticas,utilidades_domesticas,8


In [37]:
#Gewicht korrigieren
# Grenze aus der Lücke Werte darüber sind um den Faktor 1.000 zu groß
GEWICHT_GRENZE_G = 100_000

zu_schwer = df_products_clean["product_weight_g"] > GEWICHT_GRENZE_G
assert (df_products_clean.loc[zu_schwer, "product_weight_g"] % 1000 == 0).all()   # alle glatte Tausender
df_products_clean.loc[zu_schwer, "product_weight_g"] = df_products_clean.loc[zu_schwer, "product_weight_g"] / 1000
print("Durch 1.000 geteilt:", zu_schwer.sum())

# Gewicht 0 g ist unmöglich -> als fehlend markieren
gewicht_null = df_products_clean["product_weight_g"] == 0
df_products_clean.loc[gewicht_null, "product_weight_g"] = np.nan
print("Gewicht 0 g auf fehlend gesetzt:", gewicht_null.sum())



Durch 1.000 geteilt: 33
Gewicht 0 g auf fehlend gesetzt: 4


## categories.csv

In [38]:
df_categories.head(10)

,product_category_name,product_category_name_english
0,agro_industria_e_comercio,agro_industry_and_commerce
1,alimentos,food
2,alimentos_bebidas,food_drink
3,artes,art
4,artes_e_artesanato,arts_and_craftmanship
5,artigos_de_festas,party_supplies
6,artigos_de_natal,christmas_supplies
7,audio,audio
8,automotivo,auto
9,bebes,baby


In [39]:
pruefe_tabelle(
 df_categories, "categories", ["product_category_name"])

namen = df_categories["product_category_name"]
print("Namen mit Leerzeichen oder Großbuchstaben:", (namen != namen.str.strip().str.lower()).sum())

fehlt_englisch = df_categories["product_category_name_english"].isna()
display(df_categories[fehlt_englisch])

Shape: (73, 2)
<class 'pandas.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          73 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 3.6 KB

Fehlende Werte:
                                fehlend  prozent
product_category_name_english        2     2.74

Doppelte Zeilen: 0
Doppelte Schlüssel ['product_category_name']: 0
Namen mit Leerzeichen oder Großbuchstaben: 0


,product_category_name,product_category_name_english
60,pc_gamer,NaN
65,portateis_cozinha_e_preparadores_de_alimentos,NaN


In [40]:
#Fehlende englische Übersetzungen ergänzen
df_categories_clean = df_categories.copy()
uebersetzung = {"pc_gamer": "pc_gamer",
                "portateis_cozinha_e_preparadores_de_alimentos": "portable_kitchen_and_food_preparers"}

fehlt_englisch = df_categories_clean["product_category_name_english"].isna()
df_categories_clean.loc[fehlt_englisch, "product_category_name_english"] = (
    df_categories_clean.loc[fehlt_englisch, "product_category_name"].map(uebersetzung))

assert df_categories_clean["product_category_name_english"].notna().all()
display(df_categories_clean[fehlt_englisch])

,product_category_name,product_category_name_english
60,pc_gamer,pc_gamer
65,portateis_cozinha_e_preparadores_de_alimentos,portable_kitchen_and_food_preparers


# 2.1 b) Erstellung der Bestelltabelle

In [61]:
# Positionen je Bestellung zusammenfassen
position_per_order = df_items_clean.groupby("order_id").agg(
    n_items= ("order_item_id", "count"),
    items_value = ("price", "sum"),
    freight_total = ("freight_value", "sum")
).reset_index()
print("Bestellungen mit Positionen:", len(position_per_order))
assert position_per_order["order_id"].is_unique

# Letzte Bewertung je Bestellung
last_review_per_order = df_reviews.sort_values(["review_answer_timestamp", "review_id"]).groupby("order_id").tail(1).reset_index(drop=True)
print("Bestellungen mit Reviews:", len(last_review_per_order))
assert last_review_per_order["order_id"].is_unique

Bestellungen mit Positionen: 98666
Bestellungen mit Reviews: 98673


In [62]:
# Bestellungen mit Kunden verbinden
n_orders = len(df_orders)

orders_table = df_orders.merge(
    df_customers, on="customer_id", how="left", validate="one_to_one")
assert len(orders_table) == n_orders
print("Bestellungen nach Join mit Kunden:", len(orders_table))

# Je Bestellung jeder beteiligte Händler einmal (mehrere Positionen desselben Händlers zählen einmal)
order_sellers = df_items_clean[["order_id", "seller_id"]].drop_duplicates()

# Bundesstaat des Händlers ergänzen
order_sellers = order_sellers.merge(
    df_sellers[["seller_id", "seller_state"]], on="seller_id", how="left", validate="many_to_one")

# Bundesstaat des Kunden ergänzen
order_sellers = order_sellers.merge(orders_table[["order_id", "customer_state"]], on="order_id", how="left", validate="many_to_one")

# Positionen ergänzen
orders_table = orders_table.merge(position_per_order, on="order_id", how="left", validate="one_to_one")
assert len(orders_table) == n_orders

# Letzte Bewertung ergänzen
orders_table = orders_table.merge(last_review_per_order, on="order_id", how="left", validate="one_to_one")
assert len(orders_table) == n_orders

# Kontrolle: jede Bestellung genau einmal
assert orders_table["order_id"].is_unique
print("Bestelltabelle:", orders_table.shape)

Bestellungen nach Join mit Kunden: 99441
Bestelltabelle: (99441, 19)


In [63]:
# Filterbedingungen laut Aufgabe 2.1b
schritte = {
    "alle Bestellungen": orders_table["order_id"].notna(),
    "Status delivered": orders_table["order_status"] == "delivered",
    "tatsächlicher Liefertermin vorhanden": orders_table["order_delivered_customer_date"].notna(),
    "zugesagter Liefertermin vorhanden": orders_table["order_estimated_delivery_date"].notna(),
    "gültige Bewertung 1–5": orders_table["review_score"].between(1, 5),
}

# Filter nacheinander anwenden und verbleibende Zahl ausgeben
maske = pd.Series(True, index=orders_table.index)
for name, bedingung in schritte.items():
    maske = maske & bedingung
    print(f"{name}: {maske.sum()}")

# Analysetabelle
orders_analysis = orders_table[maske].reset_index(drop=True)
print("\nFallzahl für 3.1–3.3:", len(orders_analysis))

alle Bestellungen: 99441
Status delivered: 96478
tatsächlicher Liefertermin vorhanden: 96470
zugesagter Liefertermin vorhanden: 96470
gültige Bewertung 1–5: 95824

Fallzahl für 3.1–3.3: 95824


# 2.1 c) Erstellung der Merkmale

## Schlechte Bewertung

In [67]:
orders_table["bad_review"]=orders_table["review_score"].isin([1,2])


## Lieferdauer

In [75]:
# Lieferdauer in Tagen (mit Nachkommastellen)
orders_table["delivery_days"] = (
    orders_table["order_delivered_customer_date"] - orders_table["order_purchase_timestamp"]
).dt.total_seconds() / 86_400

#Einteilung in 4 Gruppen 
orders_table["delivery_days_group"]= pd.cut(orders_table["delivery_days"], bins=[0, 7, 14, 21, np.inf], labels=["≤ 7 Tage", "> 7–14 Tage", "> 14–21 Tage", "> 21 Tage"],)

print(orders_table["delivery_days_group"].value_counts(sort=False, dropna=False))

delivery_days_group
≤ 7 Tage        26046
> 7–14 Tage     40215
> 14–21 Tage    17713
> 21 Tage       12502
NaN              2965
Name: count, dtype: int64


## Verspätung

In [74]:
orders_table["delivery_delay"]= (orders_table["order_delivered_customer_date"].dt.normalize() > orders_table["order_estimated_delivery_date"])
print("Anteil verspätet:", round(orders_table.loc[orders_table["delivery_days"].notna(), "delivery_delay"].mean() * 100, 1), "%")

Anteil verspätet: 6.8 %


## Versand zwischen Bundesstaaten

In [77]:
# Je Bestellung: mindestens ein Händler aus einem anderen Bundesstaat?
interstate_per_order = order_sellers.groupby("order_id").agg(
    interstate=("other_state", "any")
).reset_index()
assert interstate_per_order["order_id"].is_unique

# An die Bestelltabelle anhängen
orders_table = orders_table.merge(interstate_per_order, on="order_id", how="left", validate="one_to_one")
assert len(orders_table) == n_orders

print("Anteil Bestellungen mit Händler in anderem Bundesstaat:",
      round(orders_table["interstate"].mean() * 100, 1), "%")

Anteil Bestellungen mit Händler in anderem Bundesstaat: 64.2 %


## Warenkorb

In [78]:
# Frachtanteil: Frachtkosten / (Warenwert + Frachtkosten)
orders_table["freight_share"] = (
    orders_table["freight_total"] / (orders_table["items_value"] + orders_table["freight_total"])
)

# Kontrolle: Überblick über alle Warenkorb-Merkmale
display(orders_table[["n_items", "items_value", "freight_total", "freight_share"]].describe().T)

,count,mean,std,min,25%,50%,75%,max
n_items,98666.0,1.141731,0.538452,1.00,1.000000,1.000000,1.000000,21.000000
items_value,98666.0,137.754076,210.645145,0.85,45.900000,86.900000,149.900000,13440.000000
freight_total,98666.0,22.823562,21.650909,0.00,13.850000,17.170000,24.040000,1794.960000
freight_share,98666.0,0.208804,0.125749,0.00,0.116502,0.183256,0.275463,0.955451
